# Compute Exact Decomposed Costs

Decomposes route costs into 3 additive components that sum exactly:
- **cost_calm**: Hull resistance (depends on speed through water)
- **cost_waves**: Wave added resistance (depends on speed through water and wave height)
- **cost_wind**: Wind resistance (depends on speed through wind)

Also computes "no current" ablation for calm and wave components to isolate current effects.

Output: `best_elites_decomposed_exact.geoparquet`

In [ ]:
from pathlib import Path

import geopandas as gpd
import msgpack
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from experiment_params import FORCING_SCENARIOS
from ship_routing.app.routing import RoutingResult
from ship_routing.core.config import SHIP_DEFAULT, PHYSICS_DEFAULT
from ship_routing.core.data import load_currents, load_waves, load_winds
from ship_routing.core.routes import Route, WayPoint
from load_tuning_results import filter_suspicious_routes, add_derived_features

import warnings

warnings.filterwarnings("ignore")

In [ ]:
def load_route_from_msgpack(msgpack_files, key, elite_idx=0):
    """Load a specific route from msgpack files."""
    for mf in msgpack_files:
        with open(mf, "rb") as f:
            raw_results = msgpack.unpack(f, raw=False)
        if key in raw_results:
            result = RoutingResult.from_msgpack(raw_results[key])
            route = result.elite_population.members[elite_idx].route
            fixed_waypoints = [
                WayPoint(
                    lon=wp.lon,
                    lat=wp.lat,
                    time=np.datetime64(wp.time, "ns"),
                )
                for wp in route.way_points
            ]
            return Route(way_points=tuple(fixed_waypoints))
    raise KeyError(f"Key {key} not found in any msgpack file")

## Load Best Baseline Routes

In [ ]:
gdf = gpd.read_parquet("../results/results_prelim.geoparquet")
gdf = add_derived_features(gdf)
gdf = filter_suspicious_routes(gdf)

# Filter to baseline only
gdf = gdf[gdf.forcing_scenario_name == "baseline"].copy()
print(f"Loaded {len(gdf)} baseline routes")

In [ ]:
# Select best elite per test case
gdf = gdf.reset_index()
gdf_best = (
    gdf.sort_values(by="elite_cost_absolute")
    .groupby(
        [
            "journey_speed_knots",
            "journey_name",
            "journey_time_start",
        ]
    )
    .first()
    .reset_index()
)
print(f"Selected {len(gdf_best)} best baseline routes")

## Load Forcing Data

In [ ]:
msgpack_files = sorted(Path("../results/").glob("*_with_crosseval.msgpack"))
print(f"Found {len(msgpack_files)} msgpack files")

bounds = gdf_best.total_bounds
spatial_bounds = (bounds[0] - 5, bounds[2] + 5, bounds[1] - 5, bounds[3] + 5)

In [ ]:
baseline = FORCING_SCENARIOS["baseline"]
time_start = np.datetime64("2021-01-01")
time_end = np.datetime64("2021-12-31T23:59:59")
data_prefix = Path("..")

currents = load_currents(
    data_prefix / baseline["currents_path"],
    time_start=time_start,
    time_end=time_end,
    engine=baseline["engine"],
    spatial_bounds=spatial_bounds,
)
print(f"Currents: {currents.dims}")

waves = load_waves(
    data_prefix / baseline["waves_path"],
    time_start=time_start,
    time_end=time_end,
    engine=baseline["engine"],
    spatial_bounds=spatial_bounds,
)
print(f"Waves: {waves.dims}")

winds = load_winds(
    data_prefix / baseline["winds_path"],
    time_start=time_start,
    time_end=time_end,
    engine=baseline["engine"],
    spatial_bounds=spatial_bounds,
)
print(f"Winds: {winds.dims}")

## Compute Decomposed Costs

In [ ]:
results = []

for _, row in tqdm(gdf_best.iterrows(), total=len(gdf_best), desc="Routes"):
    filename = row["filename"]
    elite_idx = int(row.get("n_elite", 0))

    try:
        route = load_route_from_msgpack(msgpack_files, filename, elite_idx)
        costs = route.cost_through_decomposed(
            current_data_set=currents,
            wind_data_set=winds,
            wave_data_set=waves,
            ship=SHIP_DEFAULT,
            physics=PHYSICS_DEFAULT,
        )
        costs["filename"] = filename
        results.append(costs)
    except Exception as e:
        print(f"Failed for {filename}: {e}")
        results.append({"filename": filename, "error": str(e)})

In [ ]:
df_costs = pd.DataFrame(results)
gdf_out = gdf_best.merge(df_costs, on="filename", how="left")

## Verify Decomposition

In [ ]:
sum_check = gdf_out.cost_calm + gdf_out.cost_waves + gdf_out.cost_wind
max_error = (sum_check - gdf_out.cost_total).abs().max()
print(f"Decomposition verification: max error = {max_error:.2e}")

In [ ]:
print("\n=== Component Fractions ===")
total = df_costs.cost_total.mean()
print(f"  calm/total:  {df_costs.cost_calm.mean() / total * 100:.1f}%")
print(f"  waves/total: {df_costs.cost_waves.mean() / total * 100:.1f}%")
print(f"  wind/total:  {df_costs.cost_wind.mean() / total * 100:.1f}%")

In [ ]:
print("\n=== Current Effects ===")
for col in ["delta_current_on_calm", "delta_current_on_waves", "delta_current_total"]:
    val = df_costs[col].mean()
    sign = "favorable" if val > 0 else "adverse"
    print(f"  {col}: {val:.4e} ({sign})")

## Save Results

In [ ]:
out_path = "../results/best_elites_decomposed_exact.geoparquet"
gdf_out.to_parquet(out_path)
print(f"Saved to {out_path}")